# 1.0 Import libraries

In [7]:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [8]:
import pandas as pd

# 2.0 Data load

In [50]:
icarda_gas_average = pd.read_csv("/content/drive/MyDrive/lmf/data/2026_09_22_gas_data_organization_for_khadija/icarda_gas_average.csv")
icarda_metadata = pd.read_csv("/content/drive/MyDrive/lmf/data/2026_09_22_gas_data_organization_for_khadija/icarda_gas_and_metadata.csv")
ciat = pd.read_csv("/content/drive/MyDrive/lmf/data/2026_09_22_gas_data_organization_for_khadija/compiled_categories_quartiles_modified_for_CIAT-8719_and_CIAT-7714_to_recalculate_ranking.csv")
ilri_gas = pd.read_csv("/content/drive/MyDrive/lmf/data/2026_09_22_gas_data_organization_for_khadija/ilri_gas.csv")
ilri_metadata = pd.read_csv("/content/drive/MyDrive/lmf/data/2026_09_22_gas_data_organization_for_khadija/ilri_metadata.csv")

# 3.0 Functions

In [10]:
# Function to remove the first row (duplicated original column names)
def remove_first_row(df):
    return df.iloc[1:, :].copy()

# Usage:
# subset_1_information_samples = remove_first_row(subset_1_information_samples)

In [73]:
# Define the mapping for functional groups
functional_group_mapping = {
    'Grasses': 'Grass',
    'grasses': 'Grass',
    'grass': 'Grass',
    'Shrub': 'Shrub_Trees',
    'Sub shrub': 'Shrub_Trees',
    'Shrub/tree': 'Shrub_Trees',
    'Shrub/Tree': 'Shrub_Trees',
    'Shrub/Trees': 'Shrub_Trees',
    'Shrub_Trees': 'Shrub_Trees',
    'Trees/shrubs': 'Shrub_Trees',
    'Trees/Shrubs': 'Shrub_Trees',
    'Tree/Shrubs': 'Shrub_Trees',
    'Sub shrub/vine': 'Shrub_Trees',
    'Tree': 'Shrub_Trees',
    'Arbustiva ': 'Shrub_Trees',
    'Herbáceas': 'Herbaceous_legumes',
    'Herbáceas ': 'Herbaceous_legumes',
    'Herbaceous': 'Herbaceous_legumes',
    'Herbaceous legume': 'Herbaceous_legumes',
    'Climber': 'Herbaceous_legumes',
    'Legume': 'Herbaceous_legumes'
}

# Usage:
#gas['functional_group'] = gas['functional_group'].replace(functional_group_mapping)

In [76]:
# Group by subset and id, keeping categorical columns
def mean_with_replicates(df, category_columns, numeric_columns):

    # Categorical columns to retain (excluding grouping columns)
    cat_cols = [
        col for col in category_columns
        if col not in [ 'id']
    ]

    # Keep the first value of each categorical column within each group
    cat_df = (
        df.groupby([ 'id'])[cat_cols]
          .first()
    )

    # Calculate the mean of the numeric columns
    mean_df = (
        df.groupby(['id'])[numeric_columns]
          .mean()
          .round(2)
    )

    # Count the number of replicates in each group
    mean_df['n_replicates_gas'] = (
        df.groupby(['id'])
          .size()
    )

    # Combine categorical columns, means, and replicate counts
    result = (
        cat_df
        .join(mean_df)
        .reset_index()
    )

    return result

# 4.0 Formating

## 4.1 ICARDA

In [17]:
icarda_metadata.head(2)

,icarda_id,functional_group,tanoxomic_name,gb_id,genus,run,replication,dm,ndf,om,gas_g_dm_incubated,ch4_g_dm_incubated,ch4_ndf,ch4_g_digested,tddm,ch4_g_kg_dm
0,ICARDA ID,Functional group,Latin name,GB ID,Genus,Run,Replication,DM,NDF,OM,Gas /g DM Incubated,CH4/gDM incubated,CH4/NDF,Ch4 /g digested,True digestebility,CH4 (g/kg DM)
1,250100028,Legume,Medicago sativa,101045,Medicago,1,1,92.76548217,92.76548217,88.03689603,201.4431243,28.90708834,28.90708834,40.75785051,70.92397655,16.04393415


In [20]:
icarda_gas_average.head(2)

,id,trial_id,methane_intensity,tddm,ch4_8h,ch4_24h,functional_group,taxonomic_name,family
0,IGC-2012-78-4-18,NaN,14.32,78.53,14.32,17.13,Legume,1(385x 2329)xIGC 2011-61,NaN
1,IGC-2012-78-4-18,NaN,15.91,76.80,15.91,18.01,Legume,1(385x 2329)xIGC 2011-61,NaN


In [98]:
icarda_gas_average['functional_group'] = icarda_gas_average['functional_group'].replace(functional_group_mapping)

In [100]:
# Apply the mapping for functional groups
icarda_gas_average['functional_group'] = icarda_gas_average['functional_group'].replace(functional_group_mapping)

print('Updated functional group counts for subset_1_information_samples:')
display(icarda_gas_average['functional_group'].value_counts())

Updated functional group counts for subset_1_information_samples:


,count
functional_group,
Herbaceous_legumes,519
Grass,228


In [101]:
icarda_gas_average.head(2)

,id,trial_id,methane_intensity,tddm,ch4_8h,ch4_24h,functional_group,taxonomic_name,family
0,IGC-2012-78-4-18,NaN,14.32,78.53,14.32,17.13,Herbaceous_legumes,1(385x 2329)xIGC 2011-61,NaN
1,IGC-2012-78-4-18,NaN,15.91,76.80,15.91,18.01,Herbaceous_legumes,1(385x 2329)xIGC 2011-61,NaN


In [102]:
icarda_gas_average.columns

Index(['id', 'trial_id', 'methane_intensity', 'tddm', 'ch4_8h', 'ch4_24h',
       'functional_group', 'taxonomic_name', 'family'],
      dtype='object')

In [103]:
# Sort columns
icarda_2 = icarda_gas_average[["id", "taxonomic_name", "functional_group", 'ch4_8h', 'ch4_24h', 'methane_intensity', 'tddm']]
icarda_2 = icarda_2.round(2)

In [104]:
icarda_2

,id,taxonomic_name,functional_group,ch4_8h,ch4_24h,methane_intensity,tddm
0,IGC-2012-78-4-18,1(385x 2329)xIGC 2011-61,Herbaceous_legumes,14.32,17.13,14.32,78.53
1,IGC-2012-78-4-18,1(385x 2329)xIGC 2011-61,Herbaceous_legumes,15.91,18.01,15.91,76.80
2,F3-16-16-107,1348x1330,Herbaceous_legumes,14.01,16.41,14.01,81.46
3,F3-16-16-107,1348x1330,Herbaceous_legumes,17.77,18.30,17.77,74.83
4,F3-9-9-70,1904x170,Herbaceous_legumes,13.19,15.64,13.19,75.63
...,...,...,...,...,...,...,...
742,0,Clipper//WI2291*2/WI2269/5/Soufara-02/3/RM1508...,Grass,11.47,13.66,11.47,55.01
743,0,ArabiAbiad/Arar//H.spont.41-5/Tadmor/3/Zanbaki...,Grass,11.93,13.55,11.93,56.78
744,0,Tadmor//ER/Apm/3/H.spont.38-3/Akrash-01/4/Tich...,Grass,12.17,13.52,12.17,55.07
745,0,YEA389-3/YEA475-4/4/Hma-02//11012-2/CM67/3/Mar...,Grass,11.37,12.96,11.37,50.95


In [134]:
# Sort data by number using the id column, cells with string format where located at the end.
icarda_2['_id_numeric'] = pd.to_numeric(icarda_2['id'], errors='coerce')
icarda_2 = (icarda_2.sort_values('_id_numeric', na_position='last').drop(columns='_id_numeric').reset_index(drop=True))
icarda_2

,id,taxonomic_name,functional_group,ch4_8h,ch4_24h,methane_intensity,tddm
0,0,Morex,Grass,12.94,12.65,12.94,79.27
1,0,Emir - A17/1,Grass,12.81,12.74,12.81,72.24
2,0,Morex - HB2032,Grass,13.13,12.89,13.13,69.07
3,0,Zanbaka/H.spont.41-2/4/Arar/H.spont.19-15//Hml...,Grass,10.11,9.30,10.11,82.95
4,0,Sara/4/H.Spont.96-3/3/Roho//Alger/Ceres362-1-1...,Grass,12.01,11.79,12.01,82.96
...,...,...,...,...,...,...,...
742,S30-B1-A-C2,S30-B1-A-C2,Herbaceous_legumes,11.18,14.04,44.47,65.75
743,S30-B1-B-C1,S30-B1-B-C1,Herbaceous_legumes,10.94,13.67,34.14,75.91
744,S60-B2-A-C1,S60-B2-A-C1,Herbaceous_legumes,11.37,14.26,39.65,76.26
745,S60-B4-B-C3,S60-B4-B-C3,Herbaceous_legumes,10.91,13.96,39.76,67.70


In [144]:
icarda_2 = icarda_2.rename(columns={
    'taxonomic_name': 'tax_name',
    'ch4_8h': 'ch4_percentage_in_gas_8h',
    'ch4_24h': 'ch4_percentage_in_gas_24h'
})

In [145]:
icarda_2.to_csv('/content/drive/MyDrive/lmf/output/2026_09_22_gas_data_organization_for_khadija/icarda_gas_average.csv', index=False)

## 4.2 ILRI

In [56]:
len(ilri_gas)

3159

In [57]:
ilri_gas.head(2)

,id,taxonomic_name,ch4_8h,ch4_24h,methane_intensity,tddm
0,internal control1,wheat straw,7.3,12.8,33.31,53.79
1,internal control1,wheat straw,7.3,12.8,34.77,52.38


In [58]:
ilri_gas.columns

Index(['id', 'taxonomic_name', 'ch4_8h', 'ch4_24h', 'methane_intensity',
       'tddm'],
      dtype='object')

In [59]:
len(ilri_metadata)

733

In [60]:
ilri_metadata.columns

Index(['accession_no', 'taxonomic_name', 'functional_group'], dtype='object')

In [61]:
ilri_metdata.head(2)

,accession_no,taxonomic_name,functional_group
0,147,Lablab purpureus,Herbaceous legume
1,539,Crotalaria juncea,Herbaceous legume


In [106]:
# Apply the mapping for functional groups
ilri_metadata['functional_group'] = ilri_metadata['functional_group'].replace(functional_group_mapping)

print('Updated functional group counts for subset_1_information_samples:')
display(ilri_metadata['functional_group'].value_counts())

Updated functional group counts for subset_1_information_samples:


,count
functional_group,
Herbaceous_legumes,485
Shrub_Trees,122
Grass,122


In [107]:
# To verify if identifiers of accession_no contains more than one functional group.
conflicts = (ilri_metadata.groupby('accession_no')['functional_group'].nunique())
conflicts = conflicts[conflicts > 1]
conflicts

,functional_group
accession_no,


In [108]:
# Como algunos identificadores tuvieron más de un functional_group, vamos a procesar los id que tienen uno solo functional_group asignandolo a la tabla
# A Los que tienen más de uno, se les asignará la palabra "conflict"


# Identificar accession_no con más de un functional_group
functional_groups = (ilri_metadata.groupby('accession_no')['functional_group'].agg(lambda x: x.dropna().unique()))

# Crear diccionario para asignar functional_group
functional_group_dict = {}

for accession_no, groups in functional_groups.items():

    if len(groups) == 1:
        functional_group_dict[accession_no] = groups[0]

    elif len(groups) > 1:
        functional_group_dict[accession_no] = 'conflict'

# Asignar functional_group a ilri_gas
ilri_gas['functional_group'] = (ilri_gas['id'].map(functional_group_dict))

In [109]:
print(ilri_gas['functional_group'].value_counts(dropna=False))

functional_group
Herbaceous_legumes    1953
Shrub_Trees            498
Grass                  438
NaN                    270
Name: count, dtype: int64


In [110]:
# Apparently they are not conflicts
ilri_gas[ilri_gas['functional_group'] == 'conflict']

,id,taxonomic_name,ch4_8h,ch4_24h,methane_intensity,tddm,functional_group


In [111]:
ilri_gas_2 = ilri_gas[["id", "taxonomic_name", "functional_group", 'ch4_8h', 'ch4_24h', 'methane_intensity', 'tddm']]
ilri_gas_2

,id,taxonomic_name,functional_group,ch4_8h,ch4_24h,methane_intensity,tddm
0,internal control1,wheat straw,NaN,7.3,12.8,33.31,53.79
1,internal control1,wheat straw,NaN,7.3,12.8,34.77,52.38
2,internal control1,wheat straw,NaN,7.5,12.8,36.7,50.81
3,internal control 2,Wheat bran,NaN,7.1,15.5,39.42,71.13
4,internal control 2,Wheat bran,NaN,7.1,15.9,39.41,70.88
...,...,...,...,...,...,...,...
3154,14371,Desmanthus covillei,Shrub_Trees,9.2,13.9,28.86,63.76
3155,15036,Sesbania sesban,Shrub_Trees,9,13.7,32.62,64.11
3156,15036,Sesbania sesban,Shrub_Trees,9,13.1,31.95,62.96
3157,6756,Macrotyloma axillare,Herbaceous_legumes,8.3,16.5,44.24,73.94


In [113]:
ilri_gas_2.columns

Index(['id', 'taxonomic_name', 'functional_group', 'ch4_8h', 'ch4_24h',
       'methane_intensity', 'tddm'],
      dtype='object')

In [114]:
#Columns processing
category_columns = ['id', 'taxonomic_name', 'functional_group']
numeric_columns = [ 'ch4_8h', 'ch4_24h', 'methane_intensity', 'tddm']

for df in [ilri_gas_2]:
    for col in category_columns:
        if col in df.columns:
            df[col] = df[col].astype('category')
    for col in numeric_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

In [115]:
ilri_gas_average = mean_with_replicates(ilri_gas_2, category_columns, numeric_columns)

/tmp/ipykernel_2865/2842111947.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([ 'id'])[cat_cols]
/tmp/ipykernel_2865/2842111947.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(['id'])[numeric_columns]
/tmp/ipykernel_2865/2842111947.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([ 'id'])


In [116]:
ilri_gas_average

,id,taxonomic_name,functional_group,ch4_8h,ch4_24h,methane_intensity,tddm,n_replicates_gas
0,1,Stylosanthes guianensis,Herbaceous_legumes,8.23,16.07,44.53,62.76,6
1,10,Centrosema brasilianum,Herbaceous_legumes,11.73,17.88,47.14,61.12,6
2,10000,Alysicarpus rugosus,Herbaceous_legumes,12.13,18.05,46.84,59.76,6
3,10057,Zonia latifolia,Herbaceous_legumes,8.65,15.45,39.28,54.74,6
4,10084,Mucuna pruriens,Herbaceous_legumes,9.78,15.59,40.28,61.31,12
...,...,...,...,...,...,...,...,...
389,CNPGL-93-08-1,Cenchrus purpures,Grass,10.77,16.18,49.41,59.58,6
390,Doli-I/11640,Lablab purpures,Herbaceous_legumes,9.38,15.78,41.15,62.52,12
391,Doli-II/147,Lablab purpures,Herbaceous_legumes,9.59,16.37,42.43,66.20,12
392,internal control 2,Wheat bran,NaN,8.88,15.37,45.29,69.09,135


In [124]:
# For some accessions there are more than 9 replicates used for extract the average!
# This is because, these accessions were run on multiple ocassions. So I will continue working with these averages. There is no time for curation
ilri_gas_average[ilri_gas_average['n_replicates_gas']> 9]

,id,taxonomic_name,functional_group,ch4_8h,ch4_24h,methane_intensity,tddm,n_replicates_gas
4,10084,Mucuna pruriens,Herbaceous_legumes,9.78,15.59,40.28,61.31,12
13,10379,Sesbania sesban,Shrub_Trees,6.48,17.38,33.94,73.43,12
21,10865,Sesbania sesban,Shrub_Trees,7.56,16.97,35.72,76.44,12
26,10921,Vigna parkeri,Herbaceous_legumes,9.83,17.38,45.28,67.94,12
37,11177,Calopogonium caeruleum,Shrub_Trees,10.60,18.11,40.10,69.95,12
50,11556,Cajanus cajan,Shrub_Trees,7.66,15.32,39.74,65.29,18
53,11575,Cajanus cajan,Shrub_Trees,7.15,16.73,38.39,63.76,15
57,11652,Leucaena diversifolia,Shrub_Trees,4.99,11.21,23.56,60.78,12
61,12041,Stylosanthes macrocephala,Herbaceous_legumes,9.45,16.69,43.65,61.34,12
75,12528,Trifolium tembense,Herbaceous_legumes,11.47,14.52,43.56,63.90,12


In [125]:
ilri_gas_average.drop(columns=['n_replicates_gas'], inplace=True)

In [126]:
ilri_gas_average

,id,taxonomic_name,functional_group,ch4_8h,ch4_24h,methane_intensity,tddm
0,1,Stylosanthes guianensis,Herbaceous_legumes,8.23,16.07,44.53,62.76
1,10,Centrosema brasilianum,Herbaceous_legumes,11.73,17.88,47.14,61.12
2,10000,Alysicarpus rugosus,Herbaceous_legumes,12.13,18.05,46.84,59.76
3,10057,Zonia latifolia,Herbaceous_legumes,8.65,15.45,39.28,54.74
4,10084,Mucuna pruriens,Herbaceous_legumes,9.78,15.59,40.28,61.31
...,...,...,...,...,...,...,...
389,CNPGL-93-08-1,Cenchrus purpures,Grass,10.77,16.18,49.41,59.58
390,Doli-I/11640,Lablab purpures,Herbaceous_legumes,9.38,15.78,41.15,62.52
391,Doli-II/147,Lablab purpures,Herbaceous_legumes,9.59,16.37,42.43,66.20
392,internal control 2,Wheat bran,NaN,8.88,15.37,45.29,69.09


In [128]:
ilri_gas_average = ilri_gas_average.sort_values('id').reset_index(drop=True)
ilri_gas_average

,id,taxonomic_name,functional_group,ch4_8h,ch4_24h,methane_intensity,tddm
0,1,Stylosanthes guianensis,Herbaceous_legumes,8.23,16.07,44.53,62.76
1,10,Centrosema brasilianum,Herbaceous_legumes,11.73,17.88,47.14,61.12
2,10000,Alysicarpus rugosus,Herbaceous_legumes,12.13,18.05,46.84,59.76
3,10057,Zonia latifolia,Herbaceous_legumes,8.65,15.45,39.28,54.74
4,10084,Mucuna pruriens,Herbaceous_legumes,9.78,15.59,40.28,61.31
...,...,...,...,...,...,...,...
389,CNPGL-93-08-1,Cenchrus purpures,Grass,10.77,16.18,49.41,59.58
390,Doli-I/11640,Lablab purpures,Herbaceous_legumes,9.38,15.78,41.15,62.52
391,Doli-II/147,Lablab purpures,Herbaceous_legumes,9.59,16.37,42.43,66.20
392,internal control 2,Wheat bran,NaN,8.88,15.37,45.29,69.09


In [132]:
# Sort data by number using the id column, cells with string format where located at the end.
ilri_gas_average['_id_numeric'] = pd.to_numeric(ilri_gas_average['id'], errors='coerce')
ilri_gas_average = (ilri_gas_average.sort_values('_id_numeric', na_position='last').drop(columns='_id_numeric').reset_index(drop=True))
ilri_gas_average

,id,taxonomic_name,functional_group,ch4_8h,ch4_24h,methane_intensity,tddm
0,1,Stylosanthes guianensis,Herbaceous_legumes,8.23,16.07,44.53,62.76
1,10,Centrosema brasilianum,Herbaceous_legumes,11.73,17.88,47.14,61.12
2,39,Vigna oblongifolia var. parviflora,Herbaceous_legumes,9.12,16.57,34.75,61.68
3,52,Vigna oblongifolia var. parviflora,Herbaceous_legumes,9.02,18.17,38.96,72.02
4,70,Leucaena leucocephala,Shrub_Trees,7.70,13.25,28.33,63.14
...,...,...,...,...,...,...,...
389,CNPGL-93-08-1,Cenchrus purpures,Grass,10.77,16.18,49.41,59.58
390,Doli-I/11640,Lablab purpures,Herbaceous_legumes,9.38,15.78,41.15,62.52
391,Doli-II/147,Lablab purpures,Herbaceous_legumes,9.59,16.37,42.43,66.20
392,internal control 2,Wheat bran,NaN,8.88,15.37,45.29,69.09


In [146]:
ilri_gas_average = ilri_gas_average.rename(columns={
    'taxonomic_name': 'tax_name',
    'ch4_8h': 'ch4_percentage_in_gas_8h',
    'ch4_24h': 'ch4_percentage_in_gas_24h'
})

In [147]:
ilri_gas_average.to_csv('/content/drive/MyDrive/lmf/output/2026_09_22_gas_data_organization_for_khadija/ilri_gas_average.csv', index=False)

## 4.3 CIAT

In [141]:
ciat.head(2)

,id_lab,id,subset,no,requisitioner,tax_name,functional_group,n_replicates_nutrition,dm_percentage,ash_dm,...,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,methane_intensity,tddm,ch4_category,tddm_category,lmf_category,lmf_category_rank,quartile,quartile_rank
0,F24-3470,CIAT-11194,1,199.0,Genetic_bank,Stylosanthes hamata,Herbaceous_legumes,2,92.89,9.61,...,15.17,18.02,46.12,59.25,Medium,Not High,Category_2,81.0,Q1,58.0
1,F24-3471,CIAT-11999,1,201.0,Genetic_bank,Stylosanthes guianensis,Herbaceous_legumes,2,93.50,10.46,...,12.98,16.25,46.61,58.55,Medium,Not High,Category_2,88.0,Q1,66.0


In [148]:
ciat_gas_average = ciat[["id", "tax_name", "functional_group", 'ch4_percentage_in_gas_8h', 'ch4_percentage_in_gas_24h', 'methane_intensity', 'tddm']]
ciat_gas_average.head(2)

,id,tax_name,functional_group,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,methane_intensity,tddm
0,CIAT-11194,Stylosanthes hamata,Herbaceous_legumes,15.17,18.02,46.12,59.25
1,CIAT-11999,Stylosanthes guianensis,Herbaceous_legumes,12.98,16.25,46.61,58.55


In [150]:
len(ciat_gas_average)

693

In [152]:
# These CIAT samples were delete them, because they are outliers for methane intensity. This must be corrected for next time.
ids_to_delete = ["CIAT-BH-22-3572","CIAT-BH-22-0303", "CIAT-20303"]

ciat_gas_average_2 = ciat_gas_average[~ciat_gas_average['id'].isin(ids_to_delete)].reset_index(drop=True)

In [153]:
len(ciat_gas_average_2)

690

In [156]:
ciat_gas_average_2.to_csv('/content/drive/MyDrive/lmf/output/2026_09_22_gas_data_organization_for_khadija/ciat_gas_average.csv', index=False)